# 🐉 NINE-1 — Treinamento Completo no Google Colab

Treina a **NINE-1**, uma IA de geração de código em PT-BR construída do zero em PyTorch.

**Pipeline:**
1. ✅ Clona o repositório
2. ✅ Processa dados seed e treina BPE Tokenizer
3. ✅ Pré-treina o modelo Transformer (tiny: ~10M params)
4. ✅ Testa geração de código
5. ✅ (Opcional) Fine-tuning LoRA
6. ✅ Download dos checkpoints

**Tempo estimado na T4 gratuita:** ~30min (tiny) a ~2h (small)

---
## 1. Setup — Ambiente

In [ ]:
# (1) Verifica GPU
!nvidia-smi
import torch
print('CUDA:', torch.cuda.is_available(), '-', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

# Instala dependecias (numpy ja vem no Colab)
!pip install -q numpy
print('[ok] Setup completo')

---
## 2. Clone o Repositório

In [ ]:
# (2) Clone
import os, sys
REPO = 'https://github.com/Vtgamer998/nine-1'
if not os.path.exists('nine-1'):
    !git clone {REPO}
%cd nine-1
sys.path.insert(0, '.')
print('[ok] Repositorio clonado')

---
## 3. Processar Seeds e Treinar BPE Tokenizer

In [ ]:
# (3) Processa dados seed e treina BPE
# Isso gera: corpus.txt, corpus.bin, nine1-tok.json
!python -m nine.prep_data \\
    --paths nine/data/seed \\
    --out nine/data/corpus.txt \\
    --train_bpe --vocab 4096 \\
    --bin_out nine/data/corpus.bin \\
    --tok_out nine/data/nine1-tok.json

import os
for f in os.listdir('nine/data'):
    sz = os.path.getsize(f'nine/data/{f}') if os.path.isfile(f'nine/data/{f}') else 0
    print(f'  {f}: {sz/1024:.1f} KB' if sz else f'  {f}/')
print('[ok] Dados preparados')

In [ ]:
# Testa round-trip do tokenizer
import glob
from nine.tokenizer import BPETokenizer
bt = BPETokenizer.load('nine/data/nine1-tok.json')
print(f'Tokenizer: {len(bt)} tokens')

seed_dir = 'nine/data/seed'
ok = fail = 0
for f in sorted(glob.glob(seed_dir + '/seed_*.py')):
    text = open(f).read()
    ids = bt.encode(text)
    dec = bt.decode(ids)
    if dec == text:
        ok += 1
    else:
        fail += 1
print(f'Round-trip: {ok} OK, {fail} FAIL')
print('[ok] Tokenizer OK' if fail == 0 else '[ERRO] Tokenizer com falhas!')

---
## 4. Pré-Treino do Modelo Tiny (~10M params)

Configuração: 6 layers, 6 heads, 384 embedding, contexto 256 tokens.
Roda em ~20-30 minutos na T4 gratuita.

In [ ]:
# (4) Treino tiny
!python -m nine.train \\
    --data nine/data/corpus.bin \\
    --tok nine/data/nine1-tok.json \\
    --out nine/data/nine1-base.pt \\
    --vocab 4096 \\
    --block_size 256 \\
    --n_layer 6 \\
    --n_head 6 \\
    --n_embd 384 \\
    --batch_size 64 \\
    --max_iters 2000 \\
    --lr 3e-4 \\
    --grad_accum_steps 2 \\
    --device cuda

ckpt_size = os.path.getsize('nine/data/nine1-base.pt')
print(f'Checkpoint salvo: {ckpt_size/1024/1024:.1f} MB')

---
## 5. Testar Geração de Código

In [ ]:
# (5) Testa geracao com o modelo
!python -m nine.cli \"def fibonacci(n):\" \\
    --ckpt nine/data/nine1-base.pt \\
    --tok nine/data/nine1-tok.json \\
    --tokens 80 --temp 0.8 --top_k 40

In [ ]:
# Testa modo instruct
!python -m nine.cli \"escreva uma funcao que valida email\" \\
    --mode instruct \\
    --ckpt nine/data/nine1-base.pt \\
    --tok nine/data/nine1-tok.json \\
    --tokens 120 --temp 0.8 --top_k 40

---
## 6. (Opcional) Fine-tuning LoRA

Cria dataset instrucional minimal e aplica LoRA no modelo base.

In [ ]:
# (6a) Gera dataset instrucional a partir dos seeds
import json, os

instruct_data = []
for f in sorted(glob.glob('nine/data/seed/seed_*.py')):
    with open(f) as fh:
        code = fh.read()
    lines = code.strip().split('\n')
    tasks = [l for l in lines if l.strip().startswith('#')]
    task_desc = tasks[0].lstrip('# ') if tasks else 'escreva codigo python'
    instruct_data.append({\"instruction\": task_desc, \"output\": code})

with open('nine/data/instruct.jsonl', 'w') as f:
    for item in instruct_data[:50]:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')
print(f'Dataset instrucional: {min(50, len(instruct_data))} exemplos')

In [ ]:
# (6b) Fine-tune com LoRA
# Roda rapido: ~5-10 minutos
!python -m nine.finetune \\
    --base nine/data/nine1-base.pt \\
    --data nine/data/instruct.jsonl \\
    --tok nine/data/nine1-tok.json \\
    --out nine/data/nine1-instruct.pt \\
    --lora_r 8 --lora_alpha 16 \\
    --max_iters 200 --lr 2e-4 \\
    --device cuda

lora_size = os.path.getsize('nine/data/nine1-instruct.pt')
print(f'LoRA salvo: {lora_size/1024:.1f} KB')

In [ ]:
# (6c) Testa com LoRA
!python -m nine.cli \"crie uma classe Pilha em python\" \\
    --mode instruct \\
    --ckpt nine/data/nine1-base.pt \\
    --lora nine/data/nine1-instruct.pt \\
    --tok nine/data/nine1-tok.json \\
    --tokens 150 --temp 0.7 --top_k 40

---
## 7. Download dos Checkpoints

Baixa os arquivos treinados para usar localmente (Termux, etc.)

In [ ]:
# (7) Download
from google.colab import files

print('Arquivos disponiveis para download:')
for f in ['nine/data/nine1-base.pt', 'nine/data/nine1-tok.json',
          'nine/data/nine1-instruct.pt', 'nine/data/corpus.bin']:
    if os.path.exists(f):
        sz = os.path.getsize(f)
        print(f'  {f}: {sz/1024/1024:.2f} MB' if sz > 1024*1024 else f'  {f}: {sz/1024:.1f} KB')

# Descomente para baixar:
# files.download('nine/data/nine1-base.pt')
# files.download('nine/data/nine1-tok.json')
# files.download('nine/data/nine1-instruct.pt')

---
## 8. (Opcional) DPO Alignment

Se voce tiver um dataset de preferencias, pode alinhar o modelo com DPO.

In [ ]:
# (8) DPO - exemplo com dados sinteticos
# Cria dataset de preferencias minimal
import json
prefs = []
prefs.append({
    \"prompt\": \"escreva fibonacci\",
    \"chosen\": \"def fib(n):\\n    a, b = 0, 1\\n    for _ in range(n):\\n        a, b = b, a+b\\n    return a\",
    \"rejected\": \"def fib(n):\\n    if n < 2: return n\\n    return fib(n-1) + fib(n-2)  # lento\"
})
with open('nine/data/preferences.jsonl', 'w') as f:
    for p in prefs:
        f.write(json.dumps(p, ensure_ascii=False) + '\n')
print('Dataset DPO criado (1 exemplo)')
print()
print('Para rodar DPO:')
print('  python -m nine.dpo_train --base nine/data/nine1-instruct.pt --data nine/data/preferences.jsonl --tok nine/data/nine1-tok.json --beta 0.1 --epochs 3')

---
## 9. (Opcional) Web UI no Colab

Roda a interface Gradio (compartilhada via tunnel público).

In [ ]:
# (9) Web UI no Colab
# Descomente para rodar:
# !pip install -q gradio
# !python -m nine.webui --ckpt nine/data/nine1-base.pt --tok nine/data/nine1-tok.json --share --debug

---
✅ **Treinamento concluído!**

Checkpoints salvos em `nine/data/`.

Para usar no Termux/Desktop:
```bash
# Baixe os .pt e coloque em nine/data/
python -m nine.cli "sua instrucao" --ckpt nine/data/nine1-base.pt --tok nine/data/nine1-tok.json
```